```
conda deactivate
conda remove --yes --all --name dbconnect
conda create --yes --name dbconnect python=3.7
source activate dbconnect
pip install -U databricks-connect==6.6 databricks-cli pyarrow autovizwidget
conda install --yes jupyter jupyterlab ipywidgets
```

In [ ]:
import os
import json
from pyspark.sql import SparkSession

In [ ]:
filename = os.path.expanduser(os.path.join("~",".databricks-connect"))
with open(filename) as f:
  db_connect = json.load(f)

In [ ]:
from IPython.core.magic import line_magic, line_cell_magic, Magics, magics_class

@magics_class
class DatabricksConnectMagics(Magics):

   @line_cell_magic
   def sql(self, line, cell=None):
       if cell and line:
           raise ValueError("Line must be empty for cell magic", line)
       try:
           from autovizwidget.widget.utils import display_dataframe
       except ImportError:
           print("Please run `pip install autovizwidget` to enable the visualization widget.")
           display_dataframe = lambda x: x
       return display_dataframe(self.get_spark().sql(cell or line).toPandas())

   def get_spark(self):
       user_ns = get_ipython().user_ns
       if "spark" in user_ns:
           return user_ns["spark"]
       else:
           from pyspark.sql import SparkSession
           user_ns["spark"] = SparkSession.builder.getOrCreate()
           return user_ns["spark"]

ip = get_ipython()
ip.register_magics(DatabricksConnectMagics)

In [ ]:
spark = SparkSession\
    .builder\
    .config("spark.databricks.service.server.enabled", True)\
    .config("spark.databricks.service.address", db_connect['host'])\
    .config("spark.databricks.service.token", db_connect['token'])\
    .config("spark.databricks.service.clusterId", db_connect['cluster_id'])\
    .config("spark.databricks.service.orgId", db_connect['org_id'])\
    .getOrCreate()

In [ ]:
spark\
    .range(100)\
    .selectExpr("id","rand() x","rand() y")\
    .createOrReplaceTempView("v0")

In [ ]:
%%sql

select * from v0